In [11]:
%matplotlib inline

In [12]:
import pandas as pd
from scipy.sparse import lil_matrix
import scipy.spatial.distance
import json

In [13]:
actor_name_map = {}
movie_actor_map = {}
actor_genre_map = {}


with open("imdb_movies_2000to2022.prolific.json", "r") as in_file:
    for line in in_file:

        # Read the movie on this line and parse its json
        this_movie = json.loads(line)

        # Add all actors to the id->name map
        for actor_id,actor_name in this_movie['actors']:
            actor_name_map[actor_id] = actor_name

        # For each actor, add this movie's genres to that actor's list
        for actor_id,actor_name in this_movie['actors']:
            this_actors_genres = actor_genre_map.get(actor_id, {})

            # Increment the count of genres for this actor
            for g in this_movie["genres"]:
                this_actors_genres[g] = this_actors_genres.get(g, 0) + 1

            # Update the map
            actor_genre_map[actor_id] = this_actors_genres

        # Finished with this film
        movie_actor_map[this_movie["imdb_id"]] = ({
            "movie": this_movie["title"],
            "actors": set([item[0] for item in this_movie['actors']]),
            "genres": this_movie["genres"]
        })

In [14]:
# 1. Prepare the list of genres for every movie
movie_ids = list(movie_actor_map.keys())
movie_rows = []

for m_id in movie_ids:
    # Create a dictionary where each genre of the movie has a value of 1
    genres = {g: 1 for g in movie_actor_map[m_id]['genres']}
    movie_rows.append(genres)

# 2. Create the Movie-Genre DataFrame
df_movies = pd.DataFrame(movie_rows, index=movie_ids).fillna(0)

# 3. Locate the IMDb ID for "Cars 2"
# Note: We search the movie_actor_map for the title
target_movie_title = "Cars 2"
target_movie_id = None

for m_id, info in movie_actor_map.items():
    if info['movie'] == target_movie_title:
        target_movie_id = m_id
        break

if target_movie_id:
    print(f"Found {target_movie_title} with ID: {target_movie_id}")
else:
    print("Movie not found in dataset.")

Found Cars 2 with ID: tt1216475


In [15]:
from scipy.sparse import csr_matrix

# 1. Identify all unique features
all_genres = sorted(list(df_movies.columns))
all_actors = sorted(list(actor_name_map.keys()))
feature_names = all_genres + all_actors

# Create a mapping of feature name -> column index
feature_to_idx = {feature: i for i, feature in enumerate(feature_names)}

# 2. Initialize a sparse matrix (Movies x Features)
movie_ids = list(movie_actor_map.keys())
movie_to_row = {m_id: i for i, m_id in enumerate(movie_ids)}
matrix = lil_matrix((len(movie_ids), len(feature_names)))

# 3. Fill the matrix
for m_id, info in movie_actor_map.items():
    row_idx = movie_to_row[m_id]

    # Add genres
    for g in info['genres']:
        if g in feature_to_idx:
            matrix[row_idx, feature_to_idx[g]] = 1

    # Add actors
    for a_id in info['actors']:
        if a_id in feature_to_idx:
            matrix[row_idx, feature_to_idx[a_id]] = 1

# Convert to CSR format for faster math operations
matrix_sparse = matrix.tocsr()

In [16]:
if target_movie_id:
    # Get the genre vector for Cars 2
    target_vector = df_movies.loc[target_movie_id]

    # Calculate Cosine Distance
    # (0.0 means identical genre profile, 1.0 means completely different)
    movie_distances = scipy.spatial.distance.cdist(df_movies, [target_vector], metric="cosine")[:,0]
    query_results = list(zip(df_movies.index, movie_distances))

    # Print the top 10 most similar movies
    print(f"\nMovies most similar to {target_movie_title}:")
    print("-" * 30)

    # Sort by distance (lowest is most similar)
    for sim_movie_id, sim_score in sorted(query_results, key=lambda x: x[1])[1:11]:
        movie_name = movie_actor_map[sim_movie_id]['movie']
        movie_genres = movie_actor_map[sim_movie_id]['genres']
        print(f"{movie_name} | Score: {sim_score:.4f} | Genres: {movie_genres}")


Movies most similar to Cars 2:
------------------------------
The Emperor's New Groove | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Shrek | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
The Adventures of Rocky & Bullwinkle | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
The Road to El Dorado | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Sinbad: Legend of the Seven Seas | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Monsters, Inc. | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Rugrats in Paris | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Finding Nemo | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
Ice Age | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']
The Jungle Book 2 | Score: 0.0000 | Genres: ['Adventure', 'Animation', 'Comedy']


In [17]:
target_movie_title = "Cars 2"
target_movie_id = next((m_id for m_id, info in movie_actor_map.items() if info['movie'] == target_movie_title), None)

if target_movie_id:
    target_row_idx = movie_to_row[target_movie_id]
    target_vector = matrix_sparse[target_row_idx].toarray()

    # Calculate Cosine Distance
    # Note: We convert to dense array only for the target vector to keep memory low
    distances = scipy.spatial.distance.cdist(matrix_sparse.toarray(), target_vector, metric="cosine")[:,0]

    results = []
    for i, dist in enumerate(distances):
        m_id = movie_ids[i]
        results.append((movie_actor_map[m_id]['movie'], dist, movie_actor_map[m_id]['genres']))

    # Sort and Print Top 10 (excluding the movie itself)
    print(f"Movies most similar to {target_movie_title} (Genres + Cast):")
    print("-" * 50)
    for title, score, genres in sorted(results, key=lambda x: x[1])[1:11]:
        print(f"{title: <30} | Distance: {score:.4f} | Genres: {genres}")
else:
    print("Cars 2 not found in the dataset.")

Movies most similar to Cars 2 (Genres + Cast):
--------------------------------------------------
Cars                           | Distance: 0.2857 | Genres: ['Adventure', 'Animation', 'Comedy']
Free Birds                     | Distance: 0.4286 | Genres: ['Adventure', 'Animation', 'Comedy']
Cars 3                         | Distance: 0.4286 | Genres: ['Adventure', 'Animation', 'Comedy']
Trolled                        | Distance: 0.5371 | Genres: ['Adventure', 'Animation', 'Comedy']
Turkey Day                     | Distance: 0.5636 | Genres: ['Animation', 'Comedy']
Feather Friends                | Distance: 0.5636 | Genres: ['Adventure', 'Animation']
Chicken Run                    | Distance: 0.5714 | Genres: ['Adventure', 'Animation', 'Comedy']
The Emperor's New Groove       | Distance: 0.5714 | Genres: ['Adventure', 'Animation', 'Comedy']
Shrek                          | Distance: 0.5714 | Genres: ['Adventure', 'Animation', 'Comedy']
The Adventures of Rocky & Bullwinkle | Distance: 0.5